# Stage 06 — Report

Compile all results into a LaTeX paper and PDF.
Follow `skills/stage_06.md` for detailed instructions.

In [ ]:
from paths import *
import json
import subprocess

spec      = json.loads(PAPER_SPEC.read_text())
rep_check = json.loads(REPLICATION_CHECK.read_text())
dml_res   = json.loads(DML_RESULTS.read_text())
diag      = json.loads(DIAGNOSTICS_FLAGS.read_text())

title   = spec['title']
authors = ', '.join(spec['authors'])
year    = spec['year']

print(f'Compiling report for: {title} ({year})')
print(f'Replication: {rep_check["summary"]}')
print(f'DML ({dml_res["preferred_learner"]}): coef={dml_res["preferred_coef"]}  CI=[{dml_res["preferred_ci_lo"]}, {dml_res["preferred_ci_hi"]}]')
print(f'Diagnostics: {diag["overall"]}')

In [ ]:
# --- AGENT: generate paper.tex ---
# Follow the structure in skills/stage_06.md:
#   1. Introduction
#   2. Original Paper Summary  (include table_replication.tex)
#   3. DoubleML Extension      (include table_dml.tex + forest_plot.pdf)
#   4. Diagnostics Summary
#   5. Conclusion
#   References
#
# Use \input{tables/...} and \includegraphics{figures/...}
# Write the full LaTeX source to PAPER_TEX

latex = r"""% Generated by RECAST Stage 06
\documentclass[12pt]{article}
\usepackage{booktabs,graphicx,hyperref,amsmath,geometry}
\geometry{margin=1in}

\title{RECAST: """ + title.replace('"', '\\"') + r"""\\
  \large Replication and DoubleML Extension}
\author{Original authors: """ + authors + r"""\\
  \small RECAST pipeline}
\date{}

\begin{document}
\maketitle

% AGENT: fill in each section following skills/stage_06.md

\section{Introduction}
% TODO

\section{Original Paper Summary}
\input{tables/table_replication}

\section{DoubleML Extension}
\input{tables/table_dml}
\begin{figure}[h]
  \centering
  \includegraphics[width=0.8\textwidth]{figures/forest_plot.pdf}
  \caption{Coefficient comparison: OLS/IV vs.~DML}
\end{figure}

\section{Diagnostics}
% TODO

\section{Conclusion}
% TODO

\end{document}
"""

PAPER_TEX.write_text(latex)
print(f'✓ Written: {PAPER_TEX}')

In [ ]:
# --- Compile PDF (run twice for references) ---
import os

compile_cmd = ['pdflatex', '-interaction=nonstopmode', 'paper.tex']
try:
    for run in range(2):
        result = subprocess.run(
            compile_cmd, cwd=str(PAPER_DIR),
            capture_output=True, text=True
        )
    if PAPER_PDF_OUT.exists():
        print(f'✓ Compiled: {PAPER_PDF_OUT}')
    else:
        raise RuntimeError('PDF not produced')
except (FileNotFoundError, RuntimeError) as e:
    log_path = PAPER_DIR / 'paper_compile_error.log'
    log_path.write_text(result.stdout + result.stderr)
    print(f'⚠ pdflatex failed or not available. Log: {log_path}')
    print('  paper.tex is available for manual compilation.')